# Salvador Cardoso — Week 09: Student final grades (G3)
## Question: What can we predict about a student's final mathematics grade?

This notebook examines G3 using earlier grades G1/G2 and other habits. The previous-term grades are strong predictors but are **not causal explanations**. The models are evaluated on a held-out test split. Based on the CUNY Tech Prep Week 09 syllabus.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

## 1. Read the student-performance dataset
The mathematics CSV is semicolon-delimited. If you have a local copy, it takes priority; otherwise the notebook downloads the same public UCI dataset mirror used by the class lecture.

In [ ]:
from pathlib import Path
local_candidates = [Path("student-mat.csv"), Path("data/student-mat.csv"), Path("../data/student-mat.csv")]
local_path = next((p for p in local_candidates if p.exists()), None)
url = "https://raw.githubusercontent.com/rupakc/UCI-Data-Analysis/master/Student%20Dataset/Student/student-mat.csv"
students = pd.read_csv(local_path if local_path is not None else url, sep=";")
print("Rows, columns:", students.shape)
display(students[["G1", "G2", "G3", "studytime", "failures", "absences"]].describe().round(2))
print("Missing values:", students.isna().sum().sum())

## 2. Investigate a pattern
Evaluate final grade across earlier grade groups and failures. These are associations; they do not demonstrate that changing a feature will change the final grade.

In [ ]:
display(students.groupby("failures")["G3"].agg(["count","mean","median"]).round(2))
display(students[["G1","G2","G3","studytime","failures","absences"]].corr(numeric_only=True)["G3"].sort_values(ascending=False).round(3))
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(students["G2"], students["G3"], alpha=0.4)
ax.set(xlabel="Second-period grade (G2)", ylabel="Final grade (G3)", title="Earlier grades and final grades")
plt.show()

## 3. Compare a small tree and a forest
Include G1/G2 to predict the final grade late in the term, rather than claiming the model can predict the grade at enrollment.

In [ ]:
features = ["G1", "G2", "studytime", "failures", "absences", "goout", "freetime"]
X, y = students[features], students["G3"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
tree_model = DecisionTreeRegressor(max_depth=3, random_state=42)
forest_model = RandomForestRegressor(n_estimators=200, min_samples_leaf=2, random_state=42, n_jobs=-1)
results = []
for name, model in [("Decision tree", tree_model), ("Random forest", forest_model)]:
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({"model": name, "test_R2": r2_score(y_test, pred),
                    "test_MAE": mean_absolute_error(y_test, pred),
                    "test_RMSE": np.sqrt(mean_squared_error(y_test, pred))})
display(pd.DataFrame(results).round(3))
print("Any R² > 0.9?", any(r["test_R2"] > 0.9 for r in results))

## 4. Interpret the predictions carefully

In [ ]:
pi = permutation_importance(forest_model, X_test, y_test, n_repeats=15, random_state=42, scoring="r2")
importance = pd.Series(pi.importances_mean, index=features).sort_values(ascending=False)
display(importance.to_frame("Mean decrease in held-out R²").round(3))
print("Most predictive feature in this model:", importance.index[0])

### Conclusions

- Comparing G2 with G3 tests whether recent performance is associated with final grades, not whether it causes them.
- G1 and G2 make the model suitable for **late-term** prediction; removing them tests whether study habits alone are informative.
- Review the **held-out R²**, MAE and permutation-importance results above before writing a numerical conclusion. A score above 0.9 is **not guaranteed**, and it must not be reported unless actually observed.
- This analysis is exploratory and cannot justify individual student interventions without further validation.